In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_diabetes

In [5]:
x=load_diabetes().data
y=load_diabetes().target

In [6]:
xTrain,xTest,yTrain,yTest=train_test_split(x,y,test_size=0.2,random_state=42)

In [7]:
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import r2_score
sdg=SGDRegressor(penalty='l1',max_iter=500,eta0=0.1,learning_rate='constant',alpha=0.001)
sdg.fit(xTrain,yTrain)
y_pred=sdg.predict(xTest)
print(r2_score(yTest,y_pred))
print(sdg.coef_)
print(sdg.intercept_)

0.4570544661208018
[  42.95096735 -208.53997043  518.8747236   334.43400363  -72.60961736
 -124.59384791 -215.30943353  145.70730042  385.8815859   106.65306064]
[159.94478576]


In [8]:
from sklearn.linear_model import Ridge
r=Ridge(alpha=0.1,solver='sparse_cg',max_iter=100)
r.fit(xTrain,yTrain)
y_pred=r.predict(xTest)
print(r2_score(yTest,y_pred))
print(r.coef_)
print(r.intercept_)

0.46082620941423846
[  42.92845501 -205.58997392  505.00331603  317.12107775 -108.49902293
  -86.48366689 -190.1695454   152.17171884  392.1739866    79.9151787 ]
151.4585483322631


In [13]:
class MyStochasticRidge:
    def __init__(self,learning_rate,alpha,epoch):
        self.lr=learning_rate
        self.alpha=alpha
        self.epoch=epoch
        self.intercept_=None
        self.coef_=None
    
    def fit(self,xTrain,yTrain):
        self.intercept_=0
        self.coef=np.ones(xTrain.shape[1])
        res=np.insert(self.coef,0,self.intercept_)
        xTrain=np.insert(xTrain,0,1,axis=1)
        
        for i in range(self.epoch):
            res_Der=np.dot(xTrain.T,xTrain).dot(res)-np.dot(xTrain.T,yTrain)+self.alpha*res
            res=res-self.lr*(res_Der)
        self.intercept_=res[0]
        self.coef_=res[1:]
        return (self.intercept_,self.coef_)

    def predict(self,xTest):
        return np.dot(xTest,self.coef_)+self.intercept_

In [46]:
sgdr=MyStochasticRidge(0.005,0.001,500)
sgdr.fit(xTrain,yTrain)

(np.float64(151.40454755919725),
 array([  43.37183861, -192.03766574,  496.43542567,  319.37407   ,
         -64.42788084, -113.194338  , -213.9073644 ,  144.86136322,
         367.67948022,  119.56857869]))

In [47]:
sgdr_pred=sgdr.predict(xTest)
print(r2_score(yTest,sgdr_pred))

0.45922493769364114
